In [1]:

from pyspark.sql import functions as F

# Read Delta tables
appointments = spark.table("legacy_appointments")
invoices = spark.table("legacy_invoices")

# Join and filter completed appointments
completed = (
    appointments.alias("a")
    .filter(F.col("a.status") == "Completed")
    .join(
        invoices.alias("i"),
        F.col("a.appointment_id") ==
        F.col("i.appointment_id"),
        "inner"
    )
)

# Build the monthly financial report
monthly_report = (
    completed
    .groupBy(
        F.col("a.centre_id").alias("centre_id"),
        F.date_format(
            F.col("a.appointment_date"),
            "yyyy-MM"
        ).alias("reporting_month")
    )
    .agg(
        F.count("*").alias("completed_appointments"),
        F.sum(
            F.col("i.facility_service_fee_aud")
        ).alias("facility_service_fee_aud")
    )
)

print("Monthly groups:", monthly_report.count())
display(monthly_report.orderBy(
    "centre_id", "reporting_month"
))


StatementMeta(, b2b92c55-cde0-40dd-bce7-f58408b4d458, 3, Finished, Available, Finished, False)

Monthly groups: 144


SynapseWidget(Synapse.DataFrame, 77098d67-3734-4fd7-95d3-07f0dacf0ba9)

In [2]:

# Validate before saving
assert monthly_report.count() == 144, (
    "Unexpected number of centre-month groups"
)

# Save curated Delta table
(
    monthly_report.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("curated_monthly_financial")
)

print("SUCCESS: Curated table created")
print(
    "Rows:",
    spark.table("curated_monthly_financial").count()
)


StatementMeta(, b2b92c55-cde0-40dd-bce7-f58408b4d458, 4, Finished, Available, Finished, False)

SUCCESS: Curated table created
Rows: 144


In [3]:

from pyspark.sql import functions as F

# Read baseline and new report
legacy = spark.table(
    "legacy_monthly_financial_report"
).alias("l")

curated = spark.table(
    "curated_monthly_financial"
).alias("c")

# Compare each centre-month
comparison = (
    legacy.join(
        curated,
        (F.col("l.centre_id") == F.col("c.centre_id")) &
        (F.col("l.reporting_month") ==
         F.col("c.reporting_month")),
        "full_outer"
    )
    .select(
        F.coalesce(
            F.col("l.centre_id"),
            F.col("c.centre_id")
        ).alias("centre_id"),
        F.coalesce(
            F.col("l.reporting_month"),
            F.col("c.reporting_month")
        ).alias("reporting_month"),
        F.col("l.completed_appointments")
            .alias("legacy_count"),
        F.col("c.completed_appointments")
            .alias("curated_count"),
        F.col("l.facility_service_fee_aud")
            .alias("legacy_fee"),
        F.col("c.facility_service_fee_aud")
            .alias("curated_fee")
    )
)

# Mark any missing or mismatched records
comparison = comparison.withColumn(
    "validation_result",
    F.when(
        F.col("legacy_count").isNull() |
        F.col("curated_count").isNull() |
        F.col("legacy_fee").isNull() |
        F.col("curated_fee").isNull() |
        (F.col("legacy_count") != F.col("curated_count")) |
        (
            F.abs(
                F.col("legacy_fee") - F.col("curated_fee")
            ) > F.lit(0.01)
        ),
        "FAIL"
    ).otherwise("PASS")
)

# Show summary
summary = comparison.groupBy(
    "validation_result"
).count()

display(summary)

# Show mismatches, if any
display(
    comparison.filter(
        F.col("validation_result") == "FAIL"
    )
)

# Ensure all 144 centre-month groups match
total = comparison.count()
failed = comparison.filter(
    F.col("validation_result") == "FAIL"
).count()

print("Total groups:", total)
print("Passed:", total - failed)
print("Failed:", failed)

assert total == 144 and failed == 0, (
    "Financial reconciliation did not pass"
)

print("SUCCESS: Financial reconciliation passed")


StatementMeta(, b2b92c55-cde0-40dd-bce7-f58408b4d458, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 387d7e76-48e4-45ac-b808-cab96dc2ec78)

SynapseWidget(Synapse.DataFrame, 933c8c85-d3f7-4ee7-a3a5-687804252ec8)

Total groups: 144
Passed: 144
Failed: 0
SUCCESS: Financial reconciliation passed


In [4]:

(
    comparison.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("validation_financial_reconciliation")
)

print("Validation results saved successfully")


StatementMeta(, b2b92c55-cde0-40dd-bce7-f58408b4d458, 6, Finished, Available, Finished, False)

Validation results saved successfully
